In [1]:
!pip install openai vllm==0.8.4


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
pip install pyzmq==26.4


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import json
import re
from typing import List
from openai import OpenAI
from tqdm import tqdm
from transformers import AutoTokenizer
from datasets import load_from_disk
from vllm import LLM, SamplingParams

INFO 12-13 07:37:41 [__init__.py:239] Automatically detected platform cuda.


In [2]:
vllm_model = LLM(
    model="iamjoon/qwen3-14b-text-to-sql-ko-checkpoint-700",
    dtype="bfloat16",
)

WARNING 12-13 07:37:43 [config.py:2836] Casting torch.float16 to torch.bfloat16.
INFO 12-13 07:37:50 [config.py:689] This model supports multiple tasks: {'classify', 'generate', 'score', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 12-13 07:37:50 [config.py:1901] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 12-13 07:37:52 [core.py:61] Initializing a V1 LLM engine (v0.8.4) with config: model='iamjoon/qwen3-14b-text-to-sql-ko-checkpoint-700', speculative_config=None, tokenizer='iamjoon/qwen3-14b-text-to-sql-ko-checkpoint-700', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding

Loading safetensors checkpoint shards:   0% Completed | 0/6 [00:00<?, ?it/s]


INFO 12-13 07:38:47 [loader.py:458] Loading weights took 52.05 seconds
INFO 12-13 07:38:47 [gpu_model_runner.py:1291] Model loading took 27.5185 GiB and 52.600246 seconds
INFO 12-13 07:38:57 [backends.py:416] Using cache directory: /root/.cache/vllm/torch_compile_cache/42a735a644/rank_0_0 for vLLM's torch.compile
INFO 12-13 07:38:57 [backends.py:426] Dynamo bytecode transform time: 10.35 s
INFO 12-13 07:39:04 [backends.py:132] Cache the graph of shape None for later use
INFO 12-13 07:39:44 [backends.py:144] Compiling a graph for general shape takes 45.11 s
INFO 12-13 07:40:25 [monitor.py:33] torch.compile takes 55.46 s in total
INFO 12-13 07:40:25 [kv_cache_utils.py:634] GPU KV cache size: 274,048 tokens
INFO 12-13 07:40:25 [kv_cache_utils.py:637] Maximum concurrency for 40,960 tokens per request: 6.69x
INFO 12-13 07:40:56 [gpu_model_runner.py:1626] Graph capturing finished in 30 secs, took 0.62 GiB
INFO 12-13 07:40:56 [core.py:163] init engine (profile, create kv cache, warmup model) 

In [3]:
from datasets import load_dataset, Dataset
import random

# 시드 고정
SEED = 42
random.seed(SEED)

# 1. 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/manufacturing-text-to-sql", split="train")
print(f"원본 데이터 개수: {len(dataset)}")

# 2. train/test split
test_ratio = 0.2
indices = list(range(len(dataset)))
random.shuffle(indices)
test_size = int(len(indices) * test_ratio)
test_indices = indices[:test_size]
train_indices = indices[test_size:]

# 3. OpenAI format 변환 함수
def format_data(sample):
    return {
        "messages": [
            {"role": "system", "content": sample["system_prompt"]},
            {"role": "user", "content": sample["user_prompt"]},
            {"role": "assistant", "content": sample["assistant"]},
        ]
    }

# 4. 변환
train_dataset = [format_data(dataset[i]) for i in train_indices]
test_dataset = [format_data(dataset[i]) for i in test_indices]

# 5. 결과 출력
print(f"Train: {len(train_dataset)}개")
print(f"Test:  {len(test_dataset)}개")

README.md:   0%|          | 0.00/358 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/85.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/776 [00:00<?, ? examples/s]

원본 데이터 개수: 776
Train: 621개
Test:  155개


In [4]:
# 리스트 형태에서 다시 Dataset 객체로 변경
print(type(train_dataset))
print(type(test_dataset))
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)
print(type(train_dataset))
print(type(test_dataset))

<class 'list'>
<class 'list'>
<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


In [5]:
tokenizer = AutoTokenizer.from_pretrained("iamjoon/qwen3-14b-text-to-sql-ko-checkpoint-700")

In [6]:
prompt_lst = []
label_lst = []
questions = []
contexts = []

for prompt in test_dataset["messages"]:
    text = tokenizer.apply_chat_template(
        prompt, tokenize=False, add_generation_prompt=False
    )
    input = text.split('<|im_start|>assistant')[0] + '<|im_start|>assistant'
    label = text.split('<|im_start|>assistant')[1].split('<|im_end|>')[0]
    question = text.split('<|im_start|>user')[1].split('<|im_end|>')[0].strip()
    prompt_lst.append(input)
    label_lst.append(label)
    questions.append(question)

In [7]:
questions[10]

'포르쉐라인에서 2025년 3월 오전(08~19시)과 야간(그 외) 수율을 비교해줘.'

In [8]:
label_lst[10]

"\n<think>\n\n</think>\n\nWITH FD AS (SELECT LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END AS SHIFT_GRP, SUM(TRANSQTY) AS FINISHSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='FINISH' AND DATE_FORMAT(ACTUALTIME,'%Y%m')='202503' GROUP BY LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END), SD AS (SELECT LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END AS SHIFT_GRP, SUM(TRANSQTY) AS STARTSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='START' AND DATE_FORMAT(ACTUALTIME,'%Y%m')='202503' GROUP BY LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END) SELECT FD.SHIFT_GRP, ROUND((SUM(FD.FINISHSUM)/NULLIF(SUM(SD.STARTSUM),0))*100,2) AS YIELD_PCT FROM FD INNER JOIN SD ON FD.LOTNO=SD.LOTNO AND FD.SHIFT_GRP=SD.SHIFT_GRP GROUP BY FD.SHIFT_GRP;"

In [39]:
print(prompt_lst[10])

<|im_start|>system

당신은 SQL을 생성하는 AI 모델입니다.
아래는 데이터베이스 스키마(DDL)입니다.

<SCHEMA>
-- 작업완료로그 테이블
CREATE TABLE IF NOT EXISTS `log_lottranslog` (
  `TRANSLOGID` bigint(20) NOT NULL DEFAULT 0 COMMENT '로그ID',
  `LOTNO` char(20) NOT NULL DEFAULT '' COMMENT 'LOT번호',
  `LINENO` char(20) DEFAULT NULL COMMENT '생산라인번호',
  `TRANSACTIONNAME` char(50) NOT NULL DEFAULT '' COMMENT '처리명',
  `TIMELOGGED` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '로그입력일시',
  `ACTUALTIME` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '실제실행일시',
  `MATERIALCODE` char(30) NOT NULL DEFAULT '' COMMENT '자재코드',
  `MATERIALNAME` char(50) NOT NULL DEFAULT '' COMMENT '자재명',
  `TRANSQTY` double NOT NULL DEFAULT 0 COMMENT '변경수량',
  `CURRENTQTY` double NOT NULL DEFAULT 0 COMMENT '현재수량',
  `NEXTQTY` double DEFAULT NULL COMMENT '변경반영된수량',
  `TRANSUOM` char(5) DEFAULT NULL COMMENT '측정단위',
  `WAREHOUSECODE` char(20) DEFAULT NULL COMMENT '창고코드',
  `BOPMATERIALCODE` char(30) DEFAULT NULL COMMENT '자재코드',
  `

In [41]:
print(label_lst[10])

WITH FD AS (SELECT LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END AS SHIFT_GRP, SUM(TRANSQTY) AS FINISHSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='FINISH' AND DATE_FORMAT(ACTUALTIME,'%Y%m')='202503' GROUP BY LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END), SD AS (SELECT LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END AS SHIFT_GRP, SUM(TRANSQTY) AS STARTSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='START' AND DATE_FORMAT(ACTUALTIME,'%Y%m')='202503' GROUP BY LOTNO, CASE WHEN HOUR(ACTUALTIME) BETWEEN 8 AND 19 THEN 'DAY_SHIFT' ELSE 'NIGHT_SHIFT' END) SELECT FD.SHIFT_GRP, ROUND((SUM(FD.FINISHSUM)/NULLIF(SUM(SD.STARTSUM),0))*100,2) AS YIELD_PCT FROM FD INNER JOIN SD ON FD.LOTNO=SD.LOTNO AND FD.SHIFT_GRP=SD.SHIFT_GRP GROUP BY FD.SHIFT_GRP;


In [29]:
# 생성 파라미터 설정
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=2048,
    # repetition_penalty=1.15,
)

In [30]:
preds = vllm_model.generate(prompt_lst, sampling_params)

Processed prompts:   0%|          | 0/155 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [31]:
preds = [pred.outputs[0].text for pred in preds]

In [32]:
preds[0]

"\n<think>\n\n</think>\n\nWITH FD AS (SELECT LOTNO, DATE(ACTUALTIME) AS WDATE, SUM(TRANSQTY) AS FINISHSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='FINISH' AND DATE_FORMAT(ACTUALTIME,'%Y')='2025' GROUP BY LOTNO, DATE(ACTUALTIME)), SD AS (SELECT LOTNO, DATE(ACTUALTIME) AS WDATE, SUM(TRANSQTY) AS STARTSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='START' AND DATE_FORMAT(ACTUALTIME,'%Y')='2025' GROUP BY LOTNO, DATE(ACTUALTIME)) SELECT COALESCE(F.WDATE, MIN(S.WDATE)) AS ACTUAL_DATE, ROUND((SUM(F.FINISHSUM) / NULLIF(SUM(S.STARTSUM), 0)) * 100, 2) AS CUMULATIVE_YIELD_PCT FROM FD F RIGHT JOIN SD S ON F.LOTNO = S.LOTNO AND F.WDATE = S.WDATE GROUP BY COALESCE(F.WDATE, S.WDATE) ORDER BY ACTUAL_DATE;"

In [33]:
# OpenAI API 초기화
client = OpenAI(api_key="여러분의 키 값")  # 본인의 API 키로 교체하세요

In [34]:
label_lst = [label.replace('\n<think>\n\n</think>\n\n', '') for label in label_lst]
preds = [pred.replace('\n<think>\n\n</think>\n\n', '') for pred in preds]

In [43]:
preds[0]

"WITH FD AS (SELECT LOTNO, DATE(ACTUALTIME) AS WDATE, SUM(TRANSQTY) AS FINISHSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='FINISH' AND DATE_FORMAT(ACTUALTIME,'%Y')='2025' GROUP BY LOTNO, DATE(ACTUALTIME)), SD AS (SELECT LOTNO, DATE(ACTUALTIME) AS WDATE, SUM(TRANSQTY) AS STARTSUM FROM LOG_LOTTRANSLOG WHERE LINENO='ML0000' AND TRANSACTIONNAME='START' AND DATE_FORMAT(ACTUALTIME,'%Y')='2025' GROUP BY LOTNO, DATE(ACTUALTIME)) SELECT COALESCE(F.WDATE, MIN(S.WDATE)) AS ACTUAL_DATE, ROUND((SUM(F.FINISHSUM) / NULLIF(SUM(S.STARTSUM), 0)) * 100, 2) AS CUMULATIVE_YIELD_PCT FROM FD F RIGHT JOIN SD S ON F.LOTNO = S.LOTNO AND F.WDATE = S.WDATE GROUP BY COALESCE(F.WDATE, S.WDATE) ORDER BY ACTUAL_DATE;"

## LLM 기반의 평가

In [35]:
import json
from typing import Optional


def evaluate_sql_plausibility(
    question: str,
    predicted_sql: str,
    schema: Optional[str] = None
) -> dict:
    """
    질문 + 스키마 기준 SQL 정답 합리성 평가 함수

    판단 기준:
    - 레이블과의 일치 여부는 고려하지 않음
    - 질문의 의도와 일반적인 데이터 분석 관행 기준으로
      해당 SQL이 '정답으로 채점되어도 합리적인지'를 평가
    """

    system_prompt = """
당신은 데이터베이스 SQL 질의에 대해
"질문에 대한 합리적인 정답인지"를 판단하는 전문가 평가자입니다.

중요:
- 정답 SQL은 존재하지 않습니다.
- 다른 SQL과의 일치 여부를 비교하지 마세요.
- 오직 질문, 스키마, 그리고 제시된 SQL만을 기준으로 판단하세요.

판단 기준:
- 주어진 질문의 의도를 충족하는 결과를 반환한다면 true 입니다.
- 질문에서 계산 방식이나 산식이 명확히 고정되지 않은 경우,
  일반적인 데이터 분석 관행에서 사용될 수 있는
  합리적인 가정에 기반한 SQL은 true 입니다.
- 일부 표현 방식 차이, alias, 정렬 여부는 평가에 영향을 주지 않습니다.
- 질문의 핵심 요구사항을 누락했거나,
  질문과 무관한 가정을 추가하여
  결과 의미가 명백히 달라지는 경우에만 false 입니다.

반드시 아래 JSON 형식으로만 응답하세요.
추가 설명이나 코멘트는 절대 포함하지 마세요.

{
  "is_reasonable": true 또는 false
}
""".strip()

    user_content = f"""
[질문]
{question}

[제시된 SQL]
{predicted_sql}
""".strip()

    if schema:
        user_content += f"""

[데이터베이스 스키마]
{schema}
"""

    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    try:
        result = json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        # JSON 형식 깨질 경우 보수적으로 false 처리
        result = {"is_reasonable": False}

    return {
        "is_reasonable": bool(result.get("is_reasonable", False))
    }

In [36]:
results = []

for prompt, pred in tqdm(zip(prompt_lst, preds), total=len(preds)):
    result = evaluate_sql_plausibility(
        question=prompt,
        predicted_sql=pred
    )
    results.append(result)

100%|██████████| 155/155 [01:33<00:00,  1.66it/s]


In [37]:
results[130]

{'is_reasonable': True}

In [38]:
# repetition_penalty 없음
true_count = 0
false_count = 0

for r in results:
    val = str(r["is_reasonable"]).strip().lower()
    
    if val == "true":
        true_count += 1
    elif val == "false":
        false_count += 1
    else:
        print("예외 값 발견:", r["is_reasonable"])

accuracy = true_count / len(results)

print('checkpoint-700')
print("True  개수:", true_count)
print("False 개수:", false_count)
print("Accuracy:", accuracy)

checkpoint-700
True  개수: 111
False 개수: 44
Accuracy: 0.7161290322580646


In [28]:
# repetition_penalty=1.15
true_count = 0
false_count = 0

for r in results:
    val = str(r["is_reasonable"]).strip().lower()
    
    if val == "true":
        true_count += 1
    elif val == "false":
        false_count += 1
    else:
        print("예외 값 발견:", r["is_reasonable"])

accuracy = true_count / len(results)

print('checkpoint-700')
print("True  개수:", true_count)
print("False 개수:", false_count)
print("Accuracy:", accuracy)

checkpoint-700
True  개수: 98
False 개수: 57
Accuracy: 0.632258064516129
